# Capítulo 6 – Descritores Locais e Casamento de Imagens
> **Aplicação principal:** Costura automática de panoramas

**Objetivos:**
- Entender como pontos de interesse são detectados e descritos
- Utilizar SIFT/ORB para casar pontos entre duas imagens
- Aplicar RANSAC + homografia para alinhar e mesclar imagens

**Conceitos essenciais (revisão breve):**
- Convolução (Cap. 3) – base para detectores de borda
- Transformações geométricas (Cap. 2) – homografia

---

## 6.1 Imagens utilizadas
As imagens de exemplo são duas fotos tiradas da mesma cena com deslocamento lateral:
- `left.jpg` e `right.jpg` (fornecidas no repositório, sob licença CC0)

Caso queira usar suas próprias imagens, certifique-se de que haja sobreposição de cerca de 30%.

In [3]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img_left = cv2.imread('left.jpg')
img_right = cv2.imread('right.jpg')
img_left = cv2.cvtColor(img_left, cv2.COLOR_BGR2RGB)
img_right = cv2.cvtColor(img_right, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12,4))
plt.subplot(1,2,1); plt.imshow(img_left); plt.title('Esquerda')
plt.subplot(1,2,2); plt.imshow(img_right); plt.title('Direita')
plt.show()

[ WARN:0@20.152] global loadsave.cpp:275 findDecoder imread_('left.jpg'): can't open/read file: check file path/integrity
[ WARN:0@20.152] global loadsave.cpp:275 findDecoder imread_('right.jpg'): can't open/read file: check file path/integrity


error: OpenCV(4.12.0) /private/var/folders/7k/5m3bgm8j5lg2p0szmp7jxnhw0000gn/T/pip-install-hlhsge85/opencv-python-headless_ca2b1fd92e6b4adca11f72b2e1be9efc/opencv/modules/imgproc/src/color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'


## 6.2 Detecção de pontos de interesse (SIFT)
SIFT (Scale-Invariant Feature Transform) detecta pontos que se destacam em escalas variadas e é robusto a rotações e mudanças de iluminação.

Vamos extrair e visualizar os 50 pontos mais fortes de cada imagem.

In [1]:
# Inicializar o detector SIFT (disponível no OpenCV contrib)
sift = cv2.SIFT_create()
kp_left, des_left = sift.detectAndCompute(img_left, None)
kp_right, des_right = sift.detectAndCompute(img_right, None)

# Desenhar keypoints
img_left_kp = cv2.drawKeypoints(img_left, kp_left, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
img_right_kp = cv2.drawKeypoints(img_right, kp_right, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

plt.figure(figsize=(12,6))
plt.subplot(1,2,1); plt.imshow(img_left_kp); plt.title(f'{len(kp_left)} keypoints')
plt.subplot(1,2,2); plt.imshow(img_right_kp); plt.title(f'{len(kp_right)} keypoints')
plt.show()

NameError: name 'cv2' is not defined

## 6.3 Casamento de keypoints (Brute-Force + kNN)
Comparamos os descritores da imagem esquerda com os da direita usando a distância Euclidiana. O teste da razão de Lowe (distance ratio) filtra casamentos ambíguos.

In [ ]:
bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
matches = bf.knnMatch(des_left, des_right, k=2)

# Aplicar teste da razão de Lowe
good_matches = []
for m, n in matches:
    if m.distance < 0.75 * n.distance:
        good_matches.append(m)

print(f'Casamentos brutos: {len(matches)} → casamentos bons: {len(good_matches)}')

# Visualizar os 30 melhores
img_matches = cv2.drawMatches(img_left, kp_left, img_right, kp_right, good_matches[:30], None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
plt.figure(figsize=(14,6))
plt.imshow(img_matches)
plt.title('Top 30 casamentos após Lowe ratio test')
plt.show()

## 6.4 Estimativa da homografia com RANSAC
A homografia (matriz 3×3) mapeia pontos de uma imagem para a outra. Usamos RANSAC para lidar com casamentos espúrios.

In [ ]:
# Extrair coordenadas dos pontos casados
src_pts = np.float32([kp_left[m.queryIdx].pt for m in good_matches]).reshape(-1,1,2)
dst_pts = np.float32([kp_right[m.trainIdx].pt for m in good_matches]).reshape(-1,1,2)

# Estimar homografia
H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
print(f'Matriz de homografia estimada:\n{H}')

# Aplicar transformação na imagem esquerda para alinhar com a direita
h, w = img_right.shape[:2]
img_left_warped = cv2.warpPerspective(img_left, H, (w, h))

plt.figure(figsize=(12,5))
plt.subplot(1,2,1); plt.imshow(img_left_warped); plt.title('Esquerda transformada')
plt.subplot(1,2,2); plt.imshow(img_right); plt.title('Direita original')
plt.show()

## 6.5 Costura simples das duas imagens
Podemos combinar as duas imagens em um panorama simplesmente copiando a imagem transformada sobre a direita (para sobreposição, usar média ponderada).

In [ ]:
panorama = img_right.copy()
# Região válida da imagem transformada (onde não é preta)
mask_warp = cv2.cvtColor(img_left_warped, cv2.COLOR_RGB2GRAY) > 0
panorama[mask_warp] = img_left_warped[mask_warp]

plt.figure(figsize=(14,5))
plt.imshow(panorama)
plt.title('Panorama (costura simples)')
plt.axis('off')
plt.show()

## 6.6 Exercício prático (EP)
1. Execute o código acima com um par de imagens de sua escolha (tiradas por você).
2. Altere o limiar do teste de Lowe (0.75) e observe o número de casamentos.
3. Implemente uma fusão suave (blending) na região de sobreposição usando média ponderada pela distância à borda.
4. (Avançado) Repita o processo para 3 imagens e construa um panorama mais amplo.

## Resumo do capítulo
- Pontos de interesse (SIFT/ORB) são invariantes a escala e rotação.
- O casamento por distância de descritores + razão de Lowe filtra outliers.
- RANSAC estima a homografia mesmo com muitos casamentos falsos.
- Com a homografia, podemos alinhar e combinar imagens em panoramas.

Com base no seu material já desenvolvido (PDI capítulos 1–5) e no objetivo de construir uma **Parte II de Visão Computacional (VC) com forte apelo prático**, proponho uma reorganização dos tópicos em **5 capítulos aplicados**, cada um estruturado como um caderno Jupyter (`.ipynb`) auto-contido, com **muitos exemplos reais**, **pouca teoria repetitiva** e **foco em resolver problemas concretos**.

Abaixo está a **proposta corrigida e detalhada** para os capítulos 6 a 10, que substitui o rascunho confuso do `cap03-all.ipynb`. Também incluo orientações sobre como obter **imagens livres de copyright** para todas as aplicações.

---

## ✅ Proposta dos 5 Capítulos de VC (Aplicações Reais)

### 📘 Capítulo 6 – **Descritores Locais e Casamento de Imagens**  
> *“De fotos panorâmicas a objetos 3D”*

**Aplicações reais:**
- Costura de panoramas (image stitching)
- Reconhecimento de objetos em cenas desordenadas
- Rastreamento de pontos em vídeo

**Conteúdo técnico (just-in-time):**
- Pontos de interesse (Harris, FAST)
- Descritores (SIFT, ORB – foco no uso, não na matemática pesada)
- Casamento de pontos (kNN, ratio test, RANSAC)
- Homografia e alinhamento de imagens (revisão do Cap. 2)

**Exemplo prático final:**  
Construir um panorama a partir de 2–3 fotos sobrepostas tiradas pelo próprio aluno.

**Imagens livres:**  
Fotos próprias, ou dataset [Oxford Affine](http://www.robots.ox.ac.uk/~vgg/data/data-aff.html) (uso acadêmico), ou imagens do [OpenCV samples](https://github.com/opencv/opencv/tree/4.x/samples/data).

---

### 📘 Capítulo 7 – **Classificação de Imagens com Aprendizado Clássico**  
> *“Do pixel à decisão: detectando texturas e formas”*

**Aplicações reais:**
- Classificação de tecidos (texturas médicas)
- Detecção de defeitos em superfícies industriais
- Reconhecimento de dígitos manuscritos

**Conteúdo técnico:**
- Extração de características (HOG, LBP, momentos de Hu)
- Redução de dimensionalidade (PCA – opcional)
- Classificadores (k‑NN, SVM, Random Forest)
- Validação cruzada e métricas (você já tem EP de mAP)

**Exemplo prático final:**  
Classificar imagens de frutas (laranja × maçã) ou de tecidos (linho × algodão) com acurácia > 90%.

**Imagens livres:**  
[Fruits-360](https://www.kaggle.com/datasets/moltean/fruits) (Kaggle, CC BY-SA 4.0), [MNIST](http://yann.lecun.com/exdb/mnist/), [Kylberg Texture](https://www.cb.uu.se/~gustaf/texture/) (uso livre para pesquisa).

---

### 📘 Capítulo 8 – **Introdução às Redes Neurais Convolucionais (CNNs)**  
> *“Aprendendo características como humanos”*

**Aplicações reais:**
- Classificação de raças de cães/gatos
- Detecção de pneumonia em raios-X
- Reconhecimento de placas veiculares (parte de um pipeline)

**Conteúdo técnico:**
- Conceito de convolução (revisão do Cap. 3)
- Arquitetura básica: Conv2D + ReLU + Pooling + Dense
- Transfer Learning com MobileNet/ResNet (congelar camadas)
- *Data augmentation* para evitar overfitting

**Exemplo prático final:**  
Treinar (ou fine‑tune) um classificador para distinguir 3 espécies de flores com mais de 95% de acurácia.

**Imagens livres:**  
[Flowers-102](https://www.robots.ox.ac.uk/~vgg/data/flowers/102/) (CC BY‑NC‑SA), [Chest X-ray](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia) (CC BY 4.0), [Stanford Dogs](http://vision.stanford.edu/aditya86/ImageNetDogs/) (uso acadêmico).

---

### 📘 Capítulo 9 – **Visão 3D e Geometria Projetiva**  
> *“Extraindo profundidade de imagens 2D”*

**Aplicações reais:**
- Medição de objetos a partir de fotos (forense, engenharia)
- Realidade aumentada: sobrepor texto/figura em um marcador
- Reconstrução 3D de fachadas

**Conteúdo técnico:**
- Calibração de câmera (padrão xadrez)
- Geometria estéreo (disparidade, mapa de profundidade)
- Projeção 3D → 2D e vice‑versa
- (Opcional) Estrutura a partir do movimento (SfM) com OpenCV

**Exemplo prático final:**  
Calibrar a câmera do notebook, detectar um marcador ArUco e projetar um objeto 3D (cubo) sobre ele.

**Imagens livres:**  
O próprio aluno pode imprimir um tabuleiro de xadrez ou usar marcadores ArUco; imagens de [Middlebury Stereo](https://vision.middlebury.edu/stereo/data/) (uso acadêmico).

---

### 📘 Capítulo 10 – **Projeto Integrador com Aplicações Reais**  
> *“Da imagem ao relatório final”*

**Objetivo:**  
Unir todos os conceitos anteriores em um **projeto completo**, escolhido pelo aluno dentre opções bem documentadas. O capítulo serve como roteiro para o desenvolvimento, com células de apoio (leitura de dados, funções de avaliação, etc.)

**Projetos sugeridos (todos com dados livres):**

| Projeto | Habilidades envolvidas | Dataset sugerido |
|---------|------------------------|------------------|
| **OCR de placas de veículos** | Detecção (YOLO leve), segmentação, classificação (SVM/CNN) | [UFRGS License Plates](https://www.inf.ufrgs.br/~carlos/?page_id=187) (uso acadêmico) |
| **Diagnóstico de mamografia** | Filtragem (Cap. 3), descritores (Cap. 6), classificação (Cap. 7/8) | [CBIS-DDSM](https://wiki.cancerimagingarchive.net/display/Public/CBIS-DDSM) (CC BY 3.0) |
| **Reconhecimento de faces** | Detecção (Haar cascade), descritores (LBP/HOG), SVM ou CNN | [Labeled Faces in the Wild](http://vis-www.cs.umass.edu/lfw/) (uso acadêmico) |
| **Contagem de células em microscopia** | Morfologia (Cap. 3), watershed, classificação | [BBBC](https://bbbc.broadinstitute.org/) (CC BY‑NC) |

**Estrutura do capítulo:**
- Seção 1: Como escolher e planejar o projeto
- Seção 2: Organização do código (arquivos `.py` + notebook de relatório)
- Seção 3: Dicas para apresentar resultados (tabelas, figuras, métricas)
- Seção 4: Checklist de avaliação

---

## 🖼️ Orientações sobre imagens livres de copyright

Para manter o livro **público e redistribuível**, use exclusivamente imagens com licenças **CC0, CC BY, CC BY‑SA** ou **uso educacional explícito**. Evite imagens do Google Imagens ou Pinterest.

✅ **Fontes seguras:**
- [Unsplash](https://unsplash.com/) (licença Unsplash – permite uso comercial e educacional)
- [OpenCV extra datasets](https://github.com/opencv/opencv_extra/tree/4.x/testdata)
- [scikit-image sample images](https://scikit-image.org/docs/stable/auto_examples/index.html)
- [TensorFlow Datasets](https://www.tensorflow.org/datasets/catalog/overview) (ex.: `tfds.image.Cifar10`, `tfds.image.OxfordFlowers102`)
- [Kaggle](https://www.kaggle.com/datasets) – sempre verifique a licença; muitas são CC BY‑NC‑SA ou Apache 2.0
- **Síntese própria** – gere tabuleiros de xadrez, círculos, gradientes, texturas Perlin (totalmente livres)

---

## 📓 Correção e estruturação do arquivo `cap03-all.ipynb`

O arquivo atual é uma **mistura de anotações e sugestões** sem código executável. Segue uma versão **corrigida e reorganizada** que pode servir como **template para cada capítulo da Parte II**. Salve este conteúdo como `cap06-aplicado.ipynb` (e replique para os demais).


---

## ▶️ Como prosseguir a partir daqui

1. **Aproveite os EPs já existentes** (você tem do EP01_01 ao EP02_11). Muitos deles já são aplicações (homografia, métricas, etc.). Realoque os que envolvem classificação (EP01_02, EP01_03) para o Capítulo 7, e os geométricos (EP02_09–11) para o Capítulo 6 ou 9.

2. **Para os Capítulos 3 a 5 (PDI) que estão sem exercícios**, crie notebooks semelhantes ao modelo acima, focando em:
   - Cap. 3: convolução manual, filtro mediana, detecção de bordas com Sobel.
   - Cap. 4: equalização de histograma, filtro bilateral, CLAHE.
   - Cap. 5: segmentação por Otsu, watershed, k‑means.

3. **Cada capítulo da Parte II** deve ter exatamente o mesmo formato: um problema real no título, código mínimo funcional, e um EP desafiador ao final.

4. **Evite repetir teoria** – faça links diretos para seções dos capítulos anteriores (ex.: “conforme vimos na Seção 3.2, a convolução é…”). Use células de markdown com *callouts* (⚠️ **Revisão rápida**) para refrescar memória sem reescrever.

5. **Disponibilize todas as imagens utilizadas** em um repositório GitHub com licença clara (CC0 ou CC BY). Inclua um script `download_samples.py` que baixa os datasets externos sob demanda.

---

Se desejar, posso elaborar os **notebooks completos** de cada um dos 5 capítulos de VC (com código e texto) seguindo exatamente esse modelo aplicado. Basta solicitar o próximo capítulo.